# Checkpoint Evaluation: Accuracy + Human Gaze vs Model Attention

This notebook:
- Loads multiple checkpoints (by W&B run id or explicit checkpoint path)
- Recreates **train/val/test** splits using the same logic as `train_main_utils.py` using `(seed, cities, train_gaze_frac)` from each run
- Computes **accuracy** on val/test and per-city test accuracy for:
  - `paris`, `munich`, `barcelona`, `london_uk_collideoscope`, `london_uk_gov`
- Computes saliency-style metrics between **human gaze maps** and **model attention maps**:
  - AUC, sAUC, NSS, CC, EMD, SIM, KL, IG

Evaluation policy:
- No gaze injection into the model (`gaze_left/right=None`)
- No EGViT masking enabled
- Human gaze maps are used only for metrics


In [ ]:
# --- Core imports ---
import os
import sys
import glob
import json
import re
import math
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, replace
from types import SimpleNamespace
from typing import Any, Dict, Optional, Tuple, List

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy.stats import wasserstein_distance

warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- Repro ---
SEED_GLOBAL = 30
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)
torch.cuda.manual_seed_all(SEED_GLOBAL)

# --- Device selection ---
GPU_ID = 0
if torch.cuda.is_available() and torch.cuda.device_count() > GPU_ID:
    DEVICE = torch.device(f"cuda:{GPU_ID}")
else:
    DEVICE = torch.device("cpu")

print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))


In [ ]:
# --- Make project imports work ---
# Adjust REPO_ROOT if needed.
CWD = Path.cwd().resolve()
REPO_ROOT = Path("/home/csantiago").resolve()

if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Expected repo root at {REPO_ROOT} but it does not exist.")

repo_str = str(REPO_ROOT)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

def _import_project():
    # Current train.py-compatible helpers. Keep these imports anchored at repo root.
    from train_main_utils import read_data, _load_or_split, _build_model, _load_state_dict_safely
    from backbone_registry import resolve_backbone, infer_vit_grid_size
    from gaze_policy import build_gaze_config, normalize_gaze_mode
    from data import build_eval_transforms, ComparisonsDataset
    from nets.transformer import Transformer
    from nets.gaze_guidance import GuideGuidanceConfig
    from nets.egvit import EGViTConfig

    return (
        read_data,
        _load_or_split,
        _build_model,
        _load_state_dict_safely,
        resolve_backbone,
        infer_vit_grid_size,
        build_gaze_config,
        normalize_gaze_mode,
        build_eval_transforms,
        ComparisonsDataset,
        Transformer,
        GuideGuidanceConfig,
        EGViTConfig,
    )

(
    read_data,
    _load_or_split,
    _build_model,
    _load_state_dict_safely,
    resolve_backbone,
    infer_vit_grid_size,
    build_gaze_config,
    normalize_gaze_mode,
    build_eval_transforms,
    ComparisonsDataset,
    Transformer,
    GuideGuidanceConfig,
    EGViTConfig,
) = _import_project()

print("Project imports OK.")

## Configuration

Edit the paths and the `CHECKPOINTS` list.
- `wandb_run_id` loads config + metadata from local `WANDB_DIR/run-*-<id>` directories
- `checkpoint` can be an absolute path or a relative name that exists under `MODEL_DIR`


In [ ]:
# -------------------------------
# Paths
# -------------------------------
# Master comparisons file (pickle). Required for split recreation.
# Adjust if the file name differs (e.g. comparisons_df.pkl vs comparisons_df.pickle).

COMPARISONS_PKL = "/home/csantiago/splits/comparisons_df.pickle"  # Same data object used by train.py; basename controls split-cache names

DATASET_ROOT = "/home/csantiago/images/printart/subjectivesafety_images/"  # Image root passed to ComparisonsDataset
GAZE_ROOT    = "/home/csantiago/survey_eye_tracker/Eyetracker_attention_maps/864x508"  # Gaze .npy root; transformed to patch grid by build_eval_transforms

WANDB_DIR = "/home/csantiago/wandb"  # Local W&B run folders used to recover saved train.py args
MODEL_DIR = "/home/csantiago/models"  # Checkpoint root; run_id subfolders are searched for best/last models

# -------------------------------
# Cities required for reporting
# -------------------------------
REPORT_CITIES = [  # City-wise accuracy reporting only; split recreation still uses each run's saved --cities
    "paris",
    "munich",
    "barcelona",
    #"london_uk_collideoscope",
    #"london_uk_gov",
]

# -------------------------------
# Checkpoints
# -------------------------------
# Each entry may specify either:
# - wandb_run_id + checkpoint_kind ("best" or "last") to auto-resolve from MODEL_DIR
# - checkpoint (explicit path) + optional wandb_run_id for reading run args
CHECKPOINTS = [
    {
        "tag": "dinov3_baseline",
        "wandb_run_id": "4evy0w94", # val: 77.95%, test: 74.49%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_guide",
        "wandb_run_id": "79fedrh2", # val: 77.78%, test: 74.06%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_egvit",
        "wandb_run_id": "k4v5o50x", # val: 76.74%, test: 73.72%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_align_raw",
        "wandb_run_id": "fpz1g1lb", # val: 78.82%, test: 73.63%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_align_rollout",
        "wandb_run_id": "f7kn2l9r", # val: 77.95%, test: 74.40%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "clip_baseline",
        "wandb_run_id": "xu652h29", # val: 76.56%, test: 73.97%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "clip_guide",
        "wandb_run_id": "zcodtpvw", # val: 75.69%, test: 72.00%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "clip_egvit",
        "wandb_run_id": "ml6d77fd", # val: 77.08%, test: 73.12%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "clip_align_raw",
        "wandb_run_id": "pr9f7b1q", # val: 75.87%, test: 71.23%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "clip_align_rollout",
        "wandb_run_id": "sr626vmr", # val: 75.69%, test: 72.77%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "deit_baseline",
        "wandb_run_id": "qofdasfr", # val: 78.82%, test: 72.17%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "deit_guide",
        "wandb_run_id": "zcvjqrqv", # val: 77.08%, test: 69.52%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "deit_egvit",
        "wandb_run_id": "vj2mdq8c", # val: 76.39%, test: 72.43%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "deit_align_raw",
        "wandb_run_id": "yskwdqjp", # val: 78.12%, test: 72.52%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "deit_align_rollout",
        "wandb_run_id": "6sc0vhb9", # val: 77.43%, test: 71.58%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
]
# -------------------------------
# Evaluation policy
# -------------------------------
# Attention extraction mode can be overridden globally (applies to all checkpoints).
# - None means: use the mode saved in the run metadata/config
GLOBAL_ATTN_OVERRIDE = {  # Applies only to attention extraction for this notebook; does not change checkpoint weights
    "attention_mode": "raw",  # "raw" | "rollout" | None; None keeps the mode saved in the run config
    "attn_layer": -1,         # Used only when attention_mode="raw"; -1 means final transformer block
    "force_use_attn": None,   # True forces attention hooks; False disables; None enables hooks when saliency needs vit_attn
}

# Fixations extracted from patch-grid gaze maps. Used by AUC/sAUC/NSS/IG only; KL/CC/SIM/EMD use full heatmaps.
FIXATION_MODE = "nonzero"  # "nonzero"=all gaze-supported patches; "topk"=top FIX_TOPK patches; "percentile"=patches >= FIX_PERCENTILE
FIX_TOPK = 20  # Used only when FIXATION_MODE="topk"; exactly this many patch cells become fixations
FIX_PERCENTILE = 90.0  # Used only when FIXATION_MODE="percentile"; 90 means strongest 10% of patch-grid gaze values
AUC_NEG_SAMPLES = 1000  # Used only by AUC; caps non-fixation negatives, usually no effect on 14x14/16x16 patch grids

# Shuffled AUC configuration
COMPUTE_sAUC = False  # If False, sAUC is skipped/exported as NaN; True is slower because negatives come from other images
sAUC_NEG_SAMPLES = 1000  # Used only when COMPUTE_sAUC=True; caps shuffled fixation negatives sampled from other images

# DataLoader
BATCH_SIZE = 16  # Evaluation batch size; lower this if VRAM is tight
NUM_WORKERS = 4  # DataLoader workers
PIN_MEMORY = (DEVICE.type == "cuda")  # Pin host memory only for CUDA evaluation
EVAL_DROP_LAST = True  # Match train.py validation/test loaders; set False if you want every final partial batch included

# Saliency source
SALIENCY_SOURCE = "vit_attn"  # "vit_attn" is implemented here; "vit_gradcam" is intentionally not evaluated in this notebook

print("Configured checkpoints:", [c["tag"] for c in CHECKPOINTS])


## W&B run resolution + checkpoint selection

Loads:
- training args: `seed`, `cities`, `train_gaze_frac`, plus a few extra fields when available
- resolved checkpoint path


In [ ]:
def _read_json(path: str) -> Optional[dict]:
    if (not path) or (not os.path.exists(path)):
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def _find_wandb_run_files(run_id: str, wandb_dir: str) -> dict:
    pattern = os.path.join(wandb_dir, f"run-*-{run_id}")
    matches = sorted(glob.glob(pattern))

    if not matches:
        pattern2 = os.path.join(wandb_dir, f"run-*-{run_id}*")
        matches = sorted(glob.glob(pattern2))

    if not matches:
        raise FileNotFoundError(
            f"Could not find a local W&B run directory for run id '{run_id}' under '{wandb_dir}/'."
        )

    run_dir = matches[-1]
    files_dir = os.path.join(run_dir, "files")

    def _p_any(name: str) -> Optional[str]:
        for base in (files_dir, run_dir):
            p = os.path.join(base, name)
            if os.path.exists(p):
                return p
        return None

    return {
        "run_dir": run_dir,
        "files_dir": files_dir,
        "config_json": _p_any("config.json"),
        "metadata_json": _p_any("wandb-metadata.json"),
        "summary_json": _p_any("wandb-summary.json"),
        "settings_json": _p_any("wandb-settings.json"),
    }


def _load_wandb_config_json(path: str) -> dict:
    raw = _read_json(path) or {}
    cfg = {}
    for k, v in raw.items():
        if str(k).startswith("_"):
            continue
        if isinstance(v, dict) and "value" in v:
            cfg[k] = v["value"]
        else:
            cfg[k] = v
    return cfg


def _load_metadata_args_list(metadata_path: str) -> List[str]:
    meta = _read_json(metadata_path) or {}
    args_list = meta.get("args", [])
    if not isinstance(args_list, list):
        return []
    return args_list


def _train_args_from_metadata(train_cli_args: List[str]) -> SimpleNamespace:
    import argparse

    def _str2bool(v):
        if isinstance(v, bool):
            return v
        s = str(v).strip().lower()
        if s in ("1", "true", "t", "yes", "y", "on"):
            return True
        if s in ("0", "false", "f", "no", "n", "off"):
            return False
        raise argparse.ArgumentTypeError(f"Invalid boolean value: {v}")

    p = argparse.ArgumentParser(add_help=False, allow_abbrev=False)

    p.add_argument("--model", type=str)
    p.add_argument("--backbone", type=str)

    p.add_argument("--pooling", type=str)
    p.add_argument("--pool_k", type=int)

    p.add_argument("--ties", nargs="?", const=True, type=_str2bool)

    p.add_argument("--gaze_mode", type=str)
    p.add_argument("--attention_mode", type=str)
    p.add_argument("--attn_layer", type=int)
    p.add_argument("--attn_w", type=float)
    p.add_argument("--use_nobp", nargs="?", const=True, type=_str2bool)
    p.add_argument("--guide_train_only", nargs="?", const=True, type=_str2bool)
    p.add_argument("--guidance_bottleneck_dim", type=int)
    p.add_argument("--guidance_gaze_hidden_dim", type=int)
    p.add_argument("--guidance_conv_hidden_channels", type=int)
    p.add_argument("--guidance_drop_prob", type=float)
    p.add_argument("--guidance_strength", type=float)
    p.add_argument("--egvit_mask_type", type=str)
    p.add_argument("--egvit_keep_ratio", type=float)
    p.add_argument("--egvit_focus_hw", type=int, nargs=2)
    p.add_argument("--egvit_drop_prob", type=float)
    p.add_argument("--egvit_train_only", nargs="?", const=True, type=_str2bool)
    p.add_argument("--finetune", "--ft", nargs="?", const=True, type=_str2bool)
    p.add_argument("--num_ft_blocks", type=int)
    p.add_argument("--rank_dropout", type=float)
    p.add_argument("--cross_dropout", type=float)
    p.add_argument("--gaze_map_size", type=str)
    p.add_argument("--eyetracker_filter", type=str)

    p.add_argument("--seed", type=int)
    p.add_argument("--cities", type=str)
    p.add_argument("--train_gaze_frac", type=float)

    known, _unknown = p.parse_known_args(train_cli_args)
    return SimpleNamespace(**vars(known))


def _apply_cfg_to_args(dst: SimpleNamespace, cfg: dict) -> None:
    mapping = {
        "architecture_backbone": "backbone",
        "architecture_model": "model",
        "ties": "ties",
        "gaze_mode": "gaze_mode",
        "pooling": "pooling",
        "pool_k": "pool_k",
        "attention_mode": "attention_mode",
        "attn_layer": "attn_layer",
        "attn_w": "attn_w",
        "use_nobp": "use_nobp",
        "guide_train_only": "guide_train_only",
        "guidance_bottleneck_dim": "guidance_bottleneck_dim",
        "guidance_gaze_hidden_dim": "guidance_gaze_hidden_dim",
        "guidance_conv_hidden_channels": "guidance_conv_hidden_channels",
        "guidance_drop_prob": "guidance_drop_prob",
        "guidance_strength": "guidance_strength",
        "egvit_mask_type": "egvit_mask_type",
        "egvit_keep_ratio": "egvit_keep_ratio",
        "egvit_focus_hw": "egvit_focus_hw",
        "egvit_drop_prob": "egvit_drop_prob",
        "egvit_train_only": "egvit_train_only",
        "finetune_backbone": "finetune",
        "finetune": "finetune",
        "num_ft_blocks": "num_ft_blocks",
        "rank_dropout": "rank_dropout",
        "cross_dropout": "cross_dropout",
        "gaze_map_size": "gaze_map_size",
        "eyetracker_filter": "eyetracker_filter",

        "seed": "seed",
        "cities": "cities",
        "train_gaze_frac": "train_gaze_frac",
    }

    for src, dstk in mapping.items():
        if src in cfg and cfg[src] is not None:
            setattr(dst, dstk, cfg[src])


def _apply_meta_to_args(dst: SimpleNamespace, meta_args: SimpleNamespace) -> None:
    for k, v in vars(meta_args).items():
        if v is None:
            continue
        setattr(dst, k, v)


def _infer_run_display_name(run_files: dict) -> str:
    settings = _read_json(run_files.get("settings_json")) or {}
    for k in ("run_name", "name", "display_name"):
        v = settings.get(k, None)
        if isinstance(v, str) and v.strip():
            return v.strip()

    summary = _read_json(run_files.get("summary_json")) or {}
    for k in ("run_name", "name", "display_name"):
        v = summary.get(k, None)
        if isinstance(v, str) and v.strip():
            return v.strip()

    return os.path.basename(run_files.get("run_dir", "")).strip() or "no_wandb"


def _select_checkpoint_for_run(run_id: str, run_name: str, kind: str, model_dir: str) -> str:
    kind = str(kind).lower().strip()
    if kind not in ("best", "last"):
        kind = "best"

    run_id = str(run_id).strip()
    run_name = str(run_name).strip()

    rid_dir = os.path.join(model_dir, run_id)

    patterns: list[str] = []

    if kind == "best":
        patterns += [
            os.path.join(rid_dir, "best_model_*.pt"),
            os.path.join(rid_dir, "best_model_*.pth"),
            os.path.join(rid_dir, "*_best_model_*.pt"),
            os.path.join(rid_dir, "*_best_model_*.pth"),
        ]
    else:
        patterns += [
            os.path.join(rid_dir, "last_model_*.pt"),
            os.path.join(rid_dir, "last_model_*.pth"),
            os.path.join(rid_dir, "*_last_model_*.pt"),
            os.path.join(rid_dir, "*_last_model_*.pth"),
        ]

    patterns += [
        os.path.join(model_dir, f"{run_name}_{kind}_model_*.pt"),
        os.path.join(model_dir, f"{run_name}_{kind}_model_*.pth"),
    ]

    matches: list[str] = []
    for pat in patterns:
        matches.extend(glob.glob(pat))
    matches = sorted(set(matches))

    if not matches:
        raise FileNotFoundError(
            "No checkpoint matched any known pattern.\n"
            f"  run_id: {run_id}\n"
            f"  run_name: {run_name}\n"
            f"  kind: {kind}\n"
            f"  tried:\n    " + "\n    ".join(patterns)
        )

    def _parse_score_epoch(path: str) -> Tuple[float, int]:
        base = os.path.basename(path)
        stem = base[:-3] if base.endswith(".pt") else (base[:-4] if base.endswith(".pth") else base)
        toks = stem.split("_")

        nums: list[float] = []
        for tok in toks:
            try:
                nums.append(float(tok))
            except Exception:
                pass

        if kind == "best":
            if len(nums) >= 2:
                epoch = int(round(nums[-2]))
                score = float(nums[-1])
                return (score, epoch)
            if len(nums) == 1:
                return (0.0, int(round(nums[-1])))
            return (-1.0, -1)

        if len(nums) >= 1:
            return (0.0, int(round(nums[-1])))
        return (0.0, -1)

    parsed = [(p, *_parse_score_epoch(p)) for p in matches]
    parsed.sort(key=lambda x: (x[1], x[2]), reverse=True)
    return parsed[0][0]


@dataclass
class RunResolved:
    tag: str
    run_id: Optional[str]
    run_name: str
    args: SimpleNamespace
    run_files: dict
    checkpoint_path: str


def resolve_checkpoint(entry: dict) -> RunResolved:
    tag = str(entry.get("tag", "ckpt")).strip()
    run_id = entry.get("wandb_run_id", None)
    ckpt_explicit = entry.get("checkpoint", None)
    ckpt_kind = str(entry.get("checkpoint_kind", "best")).lower().strip()

    run_files = {}
    run_name = "no_wandb"
    args = SimpleNamespace(
        model="rsscnn",
        backbone="dinov3_vitb16",
        pooling="cls",
        pool_k=10,
        ties=False,
        gaze_mode="disable",
        attention_mode="raw",
        attn_layer=-1,
        attn_w=0.0,
        use_nobp=False,
        guide_train_only=True,
        guidance_bottleneck_dim=20,
        guidance_gaze_hidden_dim=30,
        guidance_conv_hidden_channels=64,
        guidance_drop_prob=0.0,
        guidance_strength=1.0,
        egvit_mask_type="separated",
        egvit_keep_ratio=0.25,
        egvit_focus_hw=(7, 7),
        egvit_drop_prob=0.0,
        egvit_train_only=True,
        finetune=False,
        num_ft_blocks=1,
        rank_dropout=0.3,
        cross_dropout=0.3,
        gaze_map_size="auto",
        eyetracker_filter="all",
        seed=5,
        cities="all",
        train_gaze_frac=None,
    )

    if run_id is not None:
        run_files = _find_wandb_run_files(str(run_id).strip(), WANDB_DIR)
        run_name = _infer_run_display_name(run_files)

        cfg = _load_wandb_config_json(run_files["config_json"]) if run_files.get("config_json") else {}
        _apply_cfg_to_args(args, cfg)

        meta_list = _load_metadata_args_list(run_files["metadata_json"]) if run_files.get("metadata_json") else []
        meta_args = _train_args_from_metadata(meta_list) if meta_list else SimpleNamespace()
        _apply_meta_to_args(args, meta_args)

    ckpt_path = None
    if ckpt_explicit is not None:
        ck = str(ckpt_explicit).strip()
        if os.path.isabs(ck) and os.path.exists(ck):
            ckpt_path = ck
        else:
            cand = os.path.join(MODEL_DIR, ck)
            if os.path.exists(cand):
                ckpt_path = cand
            else:
                hits = sorted(glob.glob(os.path.join(MODEL_DIR, "**", ck), recursive=True))
                if hits:
                    ckpt_path = hits[0]

    if ckpt_path is None:
        if run_id is None:
            raise ValueError(f"[{tag}] checkpoint not provided and wandb_run_id is None")
        ckpt_path = _select_checkpoint_for_run(str(run_id).strip(), run_name, ckpt_kind, MODEL_DIR)

    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"[{tag}] checkpoint does not exist: {ckpt_path}")

    return RunResolved(
        tag=tag,
        run_id=str(run_id).strip() if run_id is not None else None,
        run_name=str(run_name),
        args=args,
        run_files=run_files,
        checkpoint_path=ckpt_path,
    )


def _ckpt_has_gii(ckpt_path: str, device: torch.device) -> bool:
    obj = torch.load(ckpt_path, map_location=device)
    sd = obj.get("model", obj) if isinstance(obj, dict) else obj
    if not isinstance(sd, dict):
        return False
    for k in sd.keys():
        ks = str(k)
        if ks.startswith("module."):
            ks = ks[len("module."):]
        if ks.startswith("_orig_mod."):
            ks = ks[len("_orig_mod."):]
        if ks.startswith("gii_layers.") or ks.startswith("gaze_embedder."):
            return True
    return False


def _strip_prefix(sd: dict, prefix: str) -> dict:
    if isinstance(sd, dict) and sd and all(str(k).startswith(prefix) for k in sd.keys()):
        return {str(k)[len(prefix):]: v for k, v in sd.items()}
    return sd


def load_state_dict_flexible(net: torch.nn.Module, ckpt_path: str, device: torch.device) -> None:
    obj = torch.load(ckpt_path, map_location=device)
    sd = obj.get("model", obj) if isinstance(obj, dict) else obj
    if not isinstance(sd, dict):
        raise ValueError("Checkpoint format not understood")

    sd = _strip_prefix(sd, "module.")
    sd = _strip_prefix(sd, "_orig_mod.")

    missing, unexpected = net.load_state_dict(sd, strict=False)
    if missing or unexpected:
        print("[WARN] load_state_dict strict=False had mismatches")
        print("  missing(sample):", missing[:12])
        print("  unexpected(sample):", unexpected[:12])


In [ ]:
# Resolve all checkpoints and show the run-derived settings
RESOLVED: List[RunResolved] = []
for entry in CHECKPOINTS:
    rr = resolve_checkpoint(entry)
    RESOLVED.append(rr)

rows = []
for rr in RESOLVED:
    rows.append({
        "tag": rr.tag,
        "wandb_run_id": rr.run_id or "",
        "run_name": rr.run_name,
        "checkpoint": os.path.basename(rr.checkpoint_path),
        "seed": getattr(rr.args, "seed", None),
        "cities": getattr(rr.args, "cities", None),
        "train_gaze_frac": getattr(rr.args, "train_gaze_frac", None),
        "backbone": getattr(rr.args, "backbone", None),
        "model": getattr(rr.args, "model", None),
        "pooling": getattr(rr.args, "pooling", None),
        "pool_k": getattr(rr.args, "pool_k", None),
        "ties": bool(getattr(rr.args, "ties", False)),
        "attention_mode(train)": getattr(rr.args, "attention_mode", None),
        "attn_layer(train)": getattr(rr.args, "attn_layer", None),
    })

runs_df = pd.DataFrame(rows)
display(runs_df)


## Dataset load + split recreation (train_main_utils compatible)

Implements:
- city filter via `dataset` column
- ties handling to create `score_classification`
- split: train/test then train/val with the same random seeds
- optional gaze rebalancing via `train_gaze_frac`


In [ ]:
def load_comparisons_df(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Comparisons file not found: {path}")
    try:
        import pickle
        df = pickle.load(open(path, "rb"))
    except Exception:
        df = pd.read_pickle(path)

    if not isinstance(df, pd.DataFrame):
        raise ValueError("Loaded comparisons object is not a DataFrame")

    cols_we_need = [
        "score",
        "image_l", "image_r",
        "dataset",
        "has_eyetracker",
        "npy_file_l", "npy_file_r",
        "survey_id", "trial_id",
    ]
    existing_cols = [c for c in cols_we_need if c in df.columns]
    df = df[existing_cols].copy()

    # Image filename normalization
    for side in ("image_l", "image_r"):
        if side in df.columns:
            df[side] = (
                df[side]
                .astype(str)
                .apply(lambda x: x if x.lower().endswith(".jpg") else f"{x}.jpg")
            )

    # has_eyetracker normalization
    if "has_eyetracker" in df.columns:
        df["has_eyetracker"] = (
            df["has_eyetracker"]
            .replace({"True": True, "False": False, "true": True, "false": False})
            .fillna(False)
            .astype(bool)
        )
    else:
        df["has_eyetracker"] = False

    return df


def apply_city_filter(df: pd.DataFrame, cities: str) -> pd.DataFrame:
    if "dataset" not in df.columns:
        return df.copy()

    if cities is None:
        return df.copy()

    cities = str(cities).strip()
    if cities.lower() == "all":
        return df.copy()

    selected = [c.strip() for c in cities.split(",") if c.strip()]
    if not selected:
        return df.copy()

    return df[df["dataset"].isin(selected)].copy()


def apply_ties_and_labels(df: pd.DataFrame, ties: bool) -> pd.DataFrame:
    if "score" not in df.columns:
        raise ValueError("Missing required column: score")

    df = df.copy()

    if not bool(ties):
        df = df[df["score"] != 0].copy()
        df["score_classification"] = df["score"].replace({-1: 0, +1: 1}).astype(int)
    else:
        df["score_classification"] = (df["score"] + 1).astype(int)

    return df


def _boolish_series_to_bool_mask(s: pd.Series) -> pd.Series:
    return s.astype(str).str.lower().str.strip().isin(["1", "true", "t", "yes", "y"])


def split_like_train_main_utils(
    df: pd.DataFrame,
    seed: int,
    train_pct: float = 0.7,
    val_pct: float = 0.1,
    test_pct: float = 0.2,
    train_gaze_frac: Optional[float] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    total = float(train_pct + val_pct + test_pct)
    if abs(total - 1.0) > 1e-9:
        raise ValueError(f"train_pct + val_pct + test_pct must sum to 1.0, got {total}")

    X_train, X_test = train_test_split(
        df,
        test_size=float(test_pct),
        shuffle=True,
        random_state=int(seed),
    )

    remaining = 1.0 - float(test_pct)
    val_size_of_train = float(val_pct) / max(1e-12, remaining)

    X_train, X_val = train_test_split(
        X_train,
        test_size=float(val_size_of_train),
        shuffle=True,
        random_state=int(seed),
    )

    # Optional gaze rebalancing
    if train_gaze_frac is not None:
        frac = float(train_gaze_frac)
        frac = max(0.0, min(1.0, frac))

        if "has_eyetracker" in X_train.columns and "has_eyetracker" in X_val.columns and "has_eyetracker" in X_test.columns:
            X_train = X_train.copy()
            X_val = X_val.copy()
            X_test = X_test.copy()

            tr_g = _boolish_series_to_bool_mask(X_train["has_eyetracker"])
            va_g = _boolish_series_to_bool_mask(X_val["has_eyetracker"])
            te_g = _boolish_series_to_bool_mask(X_test["has_eyetracker"])

            total_gaze = int(tr_g.sum() + va_g.sum() + te_g.sum())
            if total_gaze > 0:
                desired_train_gaze = int(math.ceil(frac * float(total_gaze)))
                current_train_gaze = int(tr_g.sum())
                need = desired_train_gaze - current_train_gaze

                if need > 0:
                    val_gaze_idx = X_val.index[va_g].to_list()
                    test_gaze_idx = X_test.index[te_g].to_list()

                    if len(val_gaze_idx) > 0:
                        val_gaze_idx = X_val.loc[val_gaze_idx].sample(frac=1.0, random_state=int(seed) + 101).index.to_list()
                    if len(test_gaze_idx) > 0:
                        test_gaze_idx = X_test.loc[test_gaze_idx].sample(frac=1.0, random_state=int(seed) + 202).index.to_list()

                    def _swap_from_source(
                        X_src: pd.DataFrame,
                        src_gaze_idx: list,
                        n_take: int,
                        X_train_local: pd.DataFrame,
                        rng_seed: int,
                    ) -> Tuple[pd.DataFrame, pd.DataFrame, int]:
                        if n_take <= 0 or len(src_gaze_idx) == 0:
                            return X_train_local, X_src, 0

                        take_idx = src_gaze_idx[:n_take]
                        take_rows = X_src.loc[take_idx]
                        X_src = X_src.drop(index=take_idx)

                        tr_mask = _boolish_series_to_bool_mask(X_train_local["has_eyetracker"])
                        train_no_gaze_idx = X_train_local.index[~tr_mask].to_list()

                        if len(train_no_gaze_idx) >= n_take:
                            swap_idx = (
                                X_train_local.loc[train_no_gaze_idx]
                                .sample(n=n_take, random_state=int(rng_seed))
                                .index.to_list()
                            )
                            swap_rows = X_train_local.loc[swap_idx]
                            X_train_local = X_train_local.drop(index=swap_idx)
                            X_src = pd.concat([X_src, swap_rows], axis=0)

                        X_train_local = pd.concat([X_train_local, take_rows], axis=0)
                        return X_train_local, X_src, len(take_idx)

                    take_val = min(need, len(val_gaze_idx))
                    X_train, X_val, got_val = _swap_from_source(X_val, val_gaze_idx, take_val, X_train, int(seed) + 303)
                    need -= got_val

                    if need > 0:
                        take_test = min(need, len(test_gaze_idx))
                        X_train, X_test, got_test = _swap_from_source(X_test, test_gaze_idx, take_test, X_train, int(seed) + 404)
                        need -= got_test

    return X_train, X_val, X_test


df_all = load_comparisons_df(COMPARISONS_PKL)
print("Loaded comparisons rows:", len(df_all))
print("Available datasets (top):")
if "dataset" in df_all.columns:
    display(df_all["dataset"].value_counts().head(20))


## Model building (no injection, no masking)

- If the checkpoint contains GII parameters, the modules are instantiated so that loading succeeds.
- Forward pass never receives gaze tensors, so injection does not occur.


In [ ]:
def _resolve_effective_attention(args: SimpleNamespace, override: Optional[dict]) -> Tuple[str, int, Optional[bool]]:
    attn_mode = str(getattr(args, "attention_mode", "rollout")).lower().strip()
    attn_layer = int(getattr(args, "attn_layer", -1))

    if attn_mode == "last":
        attn_mode = "raw"

    force_use_attn = None
    if isinstance(override, dict):
        ov_mode = override.get("attention_mode", None)
        ov_layer = override.get("attn_layer", None)
        ov_force = override.get("force_use_attn", None)

        if ov_mode is not None:
            attn_mode = str(ov_mode).lower().strip()
            if attn_mode == "last":
                attn_mode = "raw"
        if ov_layer is not None:
            attn_layer = int(ov_layer)
        if ov_force is not None:
            force_use_attn = bool(ov_force)

    if attn_mode not in ("raw", "rollout"):
        raise ValueError(f"Invalid attention_mode='{attn_mode}'. Expected raw/rollout.")

    return attn_mode, attn_layer, force_use_attn


def _load_checkpoint_state(ckpt_path: str, device: torch.device) -> dict:
    obj = torch.load(ckpt_path, map_location=device)
    state = obj.get("model", obj) if isinstance(obj, dict) else obj
    if not isinstance(state, dict):
        raise ValueError(f"Checkpoint format not understood: {ckpt_path}")
    if state and all(str(k).startswith("_orig_mod.") for k in state.keys()):
        state = {str(k)[len("_orig_mod."):]: v for k, v in state.items()}
    return state


def build_model_for_checkpoint(rr: RunResolved) -> Tuple[torch.nn.Module, dict, Tuple[int, int]]:
    backbone_alias = str(getattr(rr.args, "backbone", "dinov3_vitb16"))
    is_cnn_backbone = backbone_alias.lower().strip() in {"alex", "vgg", "dense", "resnet"}
    backbone, specs = (None, {"input_size": (3, 224, 224), "img_size": 224}) if is_cnn_backbone else resolve_backbone(backbone_alias, pretrained=False, strict=True)
    gaze_grid_size = (14, 14) if is_cnn_backbone else tuple(int(x) for x in infer_vit_grid_size(backbone, specs))

    eff_mode, eff_layer, eff_force_use_attn = _resolve_effective_attention(rr.args, GLOBAL_ATTN_OVERRIDE)
    rr.args.attention_mode = str(eff_mode).lower().strip()
    rr.args.attn_layer = int(eff_layer)
    rr.args.gaze_grid_size = tuple(gaze_grid_size)

    out_size = int(specs.get("img_size", specs.get("input_size", (3, 224, 224))[-1]))
    gaze_cfg = build_gaze_config(rr.args, is_cnn_backbone=is_cnn_backbone, out_size=out_size)
    use_attn_default = (str(SALIENCY_SOURCE).lower().strip() == "vit_attn")
    use_attn = bool(eff_force_use_attn) if (eff_force_use_attn is not None) else bool(use_attn_default)
    rr.args.gaze_cfg = replace(
        gaze_cfg,
        need_attn_maps=bool(use_attn),
        compute_kl=bool(use_attn),
        use_kl_in_loss=False,
    )

    net = _build_model(rr.args, backbone, is_cnn_backbone).to(DEVICE)
    state = _load_checkpoint_state(rr.checkpoint_path, DEVICE)
    _load_state_dict_safely(net, state, strict=True)
    net.eval()

    meta = {
        "backbone": backbone_alias,
        "model": getattr(rr.args, "model", None),
        "pooling": getattr(rr.args, "pooling", None),
        "pool_k": getattr(rr.args, "pool_k", None),
        "ties": bool(getattr(rr.args, "ties", False)),
        "attention_mode": eff_mode,
        "attn_layer": eff_layer,
        "gaze_grid_size": gaze_grid_size,
        "use_attn": bool(use_attn),
        "gaze_mode": getattr(rr.args, "gaze_mode", None),
    }
    return net, specs, gaze_grid_size


def _model_wants_gaze(net) -> bool:
    """Match scripts/train_script.py: pass gaze tensors only for forward-time gaze modes."""
    cfg = getattr(net, "cfg", None)
    guidance = getattr(cfg, "guidance", None)
    egvit = getattr(cfg, "egvit", None)
    return bool(getattr(guidance, "enabled", False) or getattr(egvit, "enabled", False))


def forward_model_matching_train(net, batch: dict):
    xL = batch["image_l"].to(DEVICE, non_blocking=True)
    xR = batch["image_r"].to(DEVICE, non_blocking=True)

    if _model_wants_gaze(net):
        missing = [k for k in ("gaze_l", "gaze_r", "has_eyetracker") if k not in batch]
        if missing:
            raise KeyError(f"Batch missing {missing} while gaze-aware forward is enabled.")
        gaze_l = batch["gaze_l"].to(DEVICE, non_blocking=True)
        gaze_r = batch["gaze_r"].to(DEVICE, non_blocking=True)
        has_eye = batch["has_eyetracker"].to(DEVICE, non_blocking=True)
        return net(xL, xR, gaze_l, gaze_r, has_eye)

    return net(xL, xR)


MODELS: Dict[str, dict] = {}
for rr in RESOLVED:
    net, specs, gaze_grid_size = build_model_for_checkpoint(rr)
    MODELS[rr.tag] = {
        "run": rr,
        "net": net,
        "specs": specs,
        "gaze_grid_size": gaze_grid_size,
    }

print("Built models:", list(MODELS.keys()))


## Datasets and loaders per checkpoint

Splits are recreated per checkpoint using:
- seed = run seed
- cities = run cities
- train_gaze_frac = run train_gaze_frac

Reporting is always computed on:
- overall val and test splits
- test split per city for `REPORT_CITIES`


In [ ]:
def _effective_cities_for_run(rr: RunResolved) -> str:
    cities = getattr(rr.args, "cities", "all")
    if cities is None:
        cities = "all"
    return str(cities).strip()


_SPLIT_CACHE: Dict[tuple, Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]] = {}


def make_splits_for_run(rr: RunResolved, df_master: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    seed = int(getattr(rr.args, "seed", 5))
    ties = bool(getattr(rr.args, "ties", False))
    train_gaze_frac = getattr(rr.args, "train_gaze_frac", None)
    if train_gaze_frac is not None:
        train_gaze_frac = float(train_gaze_frac)

    cities_str = _effective_cities_for_run(rr)
    eyetracker_filter = str(getattr(rr.args, "eyetracker_filter", "all") or "all")
    cache_key = (COMPARISONS_PKL, cities_str, ties, eyetracker_filter, seed, train_gaze_frac)
    if cache_key in _SPLIT_CACHE:
        return _SPLIT_CACHE[cache_key]

    data_args = SimpleNamespace(
        comparisons=COMPARISONS_PKL,
        cities=cities_str,
        ties=ties,
        eyetracker_filter=eyetracker_filter,
    )
    df = read_data(data_args)

    X_train, X_val, X_test = _load_or_split(
        df=df,
        seed=seed,
        comparisons_path=COMPARISONS_PKL,
        splits_dir="splits",
        train_pct=0.7,
        val_pct=0.1,
        test_pct=0.2,
        load_if_exists=False,
        save_splits=False,
        train_gaze_frac=train_gaze_frac,
    )

    _SPLIT_CACHE[cache_key] = (X_train, X_val, X_test)
    return X_train, X_val, X_test


def make_loader(
    df_split: pd.DataFrame,
    *,
    specs: dict,
    gaze_grid_size: Tuple[int, int],
    enable_gaze: bool = True,
) -> DataLoader:
    tfms, _meta = build_eval_transforms(
        specs,
        gaze_grid_size=tuple(gaze_grid_size),
        enable_gaze=bool(enable_gaze),
        gaze_output="align",
    )

    ds = ComparisonsDataset(
        dataframe=df_split,
        root_dir=DATASET_ROOT,
        transform=tfms,
        gaze_root=GAZE_ROOT,
        use_gaze=bool(enable_gaze),
        use_seg=False,
        logger=None,
    )

    loader = DataLoader(
        ds,
        batch_size=int(BATCH_SIZE),
        shuffle=False,
        num_workers=int(NUM_WORKERS),
        pin_memory=bool(PIN_MEMORY),
        drop_last=bool(EVAL_DROP_LAST),
    )
    return loader


LOADERS: Dict[str, dict] = {}
for tag, pack in MODELS.items():
    rr = pack["run"]
    X_train, X_val, X_test = make_splits_for_run(rr, df_all)

    specs = pack["specs"]
    gaze_grid = pack["gaze_grid_size"]

    loaders = {
        "val": make_loader(X_val, specs=specs, gaze_grid_size=gaze_grid, enable_gaze=True),
        "test": make_loader(X_test, specs=specs, gaze_grid_size=gaze_grid, enable_gaze=True),
        "val_df": X_val,
        "test_df": X_test,
    }

    LOADERS[tag] = loaders

    print(f"[{tag}] split sizes: train={len(X_train)} val={len(X_val)} test={len(X_test)}")

print("Loaders ready.")


## Accuracy evaluation

Accuracy is computed from `logits.output` vs `score_c` (`score_classification`).


In [ ]:
@torch.no_grad()
def compute_rank_and_class_accuracy(net: torch.nn.Module, loader: DataLoader) -> Tuple[float, float]:
    r_correct = 0
    r_total = 0

    c_correct = 0
    c_total = 0
    has_logits = None

    for batch in loader:
        y_r = batch["score_r"].to(DEVICE, non_blocking=True).long()
        y_c = batch["score_c"].to(DEVICE, non_blocking=True).long()

        out = forward_model_matching_train(net, batch)

        # -----------------------
        # Ranking accuracy (matches train_script RankAccuracy semantics)
        # -----------------------
        sL = out["left"]["output"].view(-1)
        sR = out["right"]["output"].view(-1)
        diff = sL - sR

        mask = (y_r != 0)  # ignore ties in ranking
        if int(mask.sum().item()) > 0:
            r_correct += int(((-y_r[mask]) * diff[mask] > 0).sum().item())
            r_total += int(mask.sum().item())

        # -----------------------
        # Classification accuracy
        # -----------------------
        logits_pack = out.get("logits", None)
        if has_logits is None:
            has_logits = isinstance(logits_pack, dict) and ("output" in logits_pack) and (logits_pack["output"] is not None)

        if bool(has_logits):
            logits = logits_pack["output"]
            pred = torch.argmax(logits, dim=1)
            c_correct += int((pred == y_c).sum().item())
            c_total += int(y_c.numel())

    rank_acc = float(r_correct) / float(max(1, r_total)) if r_total > 0 else float("nan")
    class_acc = float(c_correct) / float(max(1, c_total)) if c_total > 0 else float("nan")
    return rank_acc, class_acc


def eval_accuracy_table_full() -> pd.DataFrame:
    """
    Paper-style accuracy table (Rank/Class).

    - Cities restricted to REPORT_CITIES (in that order).
    - Val/Test/City-Avg/City-wise are shown as ONE string per cell: "rank/class" in percent.
      Example: "77.00/76.00" means 77% rank accuracy and 76% classification accuracy.
    - Best per group (Val/Test/City Avg) highlighted (bold) if best in rank OR class.
    """

    # -------------------------
    # helpers
    # -------------------------
    def fmt_pct(x: float) -> str:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "-"
        return f"{x * 100:.2f}"

    def fmt_pair(r: float, c: float) -> str:
        if (isinstance(r, float) and np.isnan(r)) or (isinstance(c, float) and np.isnan(c)):
            return "-"
        return f"{fmt_pct(r)}/{fmt_pct(c)}"

    rows = []

    # -------------------------
    # compute numeric metrics
    # -------------------------
    for tag, pack in MODELS.items():
        rr = pack["run"]
        net = pack["net"]

        val_loader = LOADERS[tag]["val"]
        test_loader = LOADERS[tag]["test"]

        # val/test (single pass each)
        r_val, c_val = compute_rank_and_class_accuracy(net, val_loader)
        r_test, c_test = compute_rank_and_class_accuracy(net, test_loader)

        # city-wise (restricted)
        rank_city = {}
        class_city = {}

        for city in REPORT_CITIES:
            df_city = df_all[df_all["dataset"] == city].copy()
            if len(df_city) == 0:
                rank_city[city] = np.nan
                class_city[city] = np.nan
                continue

            ties = bool(getattr(rr.args, "ties", False))
            df_city = apply_ties_and_labels(df_city, ties)

            loader_city = make_loader(
                df_city,
                specs=pack["specs"],
                gaze_grid_size=pack["gaze_grid_size"],
                enable_gaze=True,
            )

            r_city, c_city = compute_rank_and_class_accuracy(net, loader_city)
            rank_city[city] = r_city
            class_city[city] = c_city

        r_city_avg = float(np.nanmean([rank_city[c] for c in REPORT_CITIES])) if len(REPORT_CITIES) else np.nan
        c_city_avg = float(np.nanmean([class_city[c] for c in REPORT_CITIES])) if len(REPORT_CITIES) else np.nan

        row = {
            ("Model", ""): tag,
            ("train_gaze_frac", ""): float(getattr(getattr(rr, "args", None), "train_gaze_frac", np.nan)),
            ("wandb_run_id", ""): (rr.run_id or ""),
        
            ("Val", "Rank"): r_val,
            ("Val", "Class"): c_val,
        
            ("Test", "Rank"): r_test,
            ("Test", "Class"): c_test,
        
            ("City Avg", "Rank"): r_city_avg,
            ("City Avg", "Class"): c_city_avg,
        }
        
        for city in REPORT_CITIES:
            row[("City-wise", f"{city} Rank")] = rank_city[city]
            row[("City-wise", f"{city} Class")] = class_city[city]

        rows.append(row)

    df_num = pd.DataFrame(rows)
    df_num.columns = pd.MultiIndex.from_tuples(df_num.columns)

    # -------------------------
    # build display table with combined "rank/class" cells
    # -------------------------
    disp_rows = []
    for i in range(len(df_num)):
        r = df_num.iloc[i]

        disp = {
            ("Model", ""): r[("Model", "")],
            ("train_gaze_frac", ""): "-" if pd.isna(r[("train_gaze_frac", "")]) else f"{float(r[('train_gaze_frac','')]):.2f}",
            ("wandb_run_id", ""): r[("wandb_run_id", "")],
        
            ("Val", "Acc (Rank/Class)"): fmt_pair(r[("Val", "Rank")], r[("Val", "Class")]),
            ("Test", "Acc (Rank/Class)"): fmt_pair(r[("Test", "Rank")], r[("Test", "Class")]),
            ("City Avg", "Acc (Rank/Class)"): fmt_pair(r[("City Avg", "Rank")], r[("City Avg", "Class")]),
        }

        for city in REPORT_CITIES:
            disp[("City-wise", city)] = fmt_pair(
                r[("City-wise", f"{city} Rank")],
                r[("City-wise", f"{city} Class")],
            )

        disp_rows.append(disp)

    df_disp = pd.DataFrame(disp_rows)
    df_disp.columns = pd.MultiIndex.from_tuples(df_disp.columns)

    # -------------------------
    # bold best per group (Val/Test/City Avg)
    # If best in rank OR class, bold whole "rank/class" cell.
    # -------------------------
    def bold_best_group(group: str):
        rank_col = (group, "Rank")
        class_col = (group, "Class")
        disp_col = (group, "Acc (Rank/Class)")

        if rank_col not in df_num.columns or class_col not in df_num.columns:
            return

        rank_vals = df_num[rank_col].astype(float).to_numpy()
        class_vals = df_num[class_col].astype(float).to_numpy()

        best_rank = np.nanmax(rank_vals) if not np.all(np.isnan(rank_vals)) else np.nan
        best_class = np.nanmax(class_vals) if not np.all(np.isnan(class_vals)) else np.nan

        for i in range(len(df_disp)):
            vr = df_num.iloc[i][rank_col]
            vc = df_num.iloc[i][class_col]

            vr = float(vr) if not pd.isna(vr) else np.nan
            vc = float(vc) if not pd.isna(vc) else np.nan

            is_best = (
                (not np.isnan(best_rank) and not np.isnan(vr) and abs(vr - best_rank) < 1e-12)
                or
                (not np.isnan(best_class) and not np.isnan(vc) and abs(vc - best_class) < 1e-12)
            )

            if is_best and df_disp.iloc[i][disp_col] != "-":
                df_disp.iat[i, df_disp.columns.get_loc(disp_col)] = f"**{df_disp.iloc[i][disp_col]}**"

    for grp in ["Val", "Test", "City Avg"]:
        bold_best_group(grp)

    # -------------------------
    # sort by Test Rank (descending)
    # -------------------------
    if ("Test", "Rank") in df_num.columns:
        sort_key = df_num[("Test", "Rank")].fillna(-np.inf).to_numpy()
        order = np.argsort(-sort_key)

        df_num = df_num.iloc[order].reset_index(drop=True)
        df_disp = df_disp.iloc[order].reset_index(drop=True)

    # -------------------------
    # final column order
    # -------------------------
    ordered = [
        ("Model", ""),
        ("train_gaze_frac", ""),
        ("wandb_run_id", ""),
        ("Val", "Acc (Rank/Class)"),
        ("Test", "Acc (Rank/Class)"),
        ("City Avg", "Acc (Rank/Class)"),
    ]
    ordered += [("City-wise", c) for c in REPORT_CITIES]

    df_disp = df_disp[ordered]

    display(df_disp)
    return df_num


#acc_df = eval_accuracy_table_full()


## Saliency metrics: human gaze vs model attention

Metrics:
- AUC (ROC AUC over all pixels; positives are gaze fixations)
- sAUC (shuffled AUC using negative fixation samples from other images)
- NSS (z-scored model map sampled at fixation locations)
- CC (Pearson correlation between z-scored maps)
- EMD (1D Wasserstein over flattened distributions)
- SIM (histogram intersection over distributions)
- KL (KL(gaze || model) over distributions)
- IG (log-likelihood gain at fixations vs uniform baseline)

Notes:
- Gaze maps may be unnormalized; normalization is applied per metric.
- Attention maps returned by the model are normalized by softmax at token level but are still re-normalized when a distribution is required.


In [ ]:
def _to_2d(x: torch.Tensor) -> torch.Tensor:
    if x is None:
        return x
    if x.ndim == 4 and x.shape[1] == 1:
        return x[:, 0]
    if x.ndim == 3:
        return x
    if x.ndim == 2:
        return x.unsqueeze(0)
    return x


def _minmax01(m: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    vmin = float(np.min(m))
    vmax = float(np.max(m))
    if vmax - vmin < eps:
        return np.zeros_like(m, dtype=np.float32)
    return ((m - vmin) / (vmax - vmin + eps)).astype(np.float32)


def _prob_norm(m: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    m = np.clip(m.astype(np.float64), 0.0, None)
    s = float(np.sum(m))
    if s <= eps:
        out = np.ones_like(m, dtype=np.float64)
        out /= float(out.size)
        return out.astype(np.float64)
    return (m / s).astype(np.float64)


def _zscore(m: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    mu = float(np.mean(m))
    sd = float(np.std(m))
    if sd < eps:
        return np.zeros_like(m, dtype=np.float32)
    return ((m - mu) / sd).astype(np.float32)


def _fixation_mask_from_gaze(gaze_map_01: np.ndarray) -> np.ndarray:
    h, w = gaze_map_01.shape
    flat = gaze_map_01.reshape(-1)
    mode = str(FIXATION_MODE).lower().strip()

    if mode == "nonzero":
        mask = (gaze_map_01 > 0).astype(np.uint8)
        if int(mask.sum()) == 0:
            mask.reshape(-1)[int(np.argmax(flat))] = 1
        return mask

    if mode == "topk":
        k = int(FIX_TOPK)
        k = max(1, min(k, flat.size))
        idx = np.argpartition(flat, -k)[-k:]
        mask = np.zeros_like(flat, dtype=np.uint8)
        mask[idx] = 1
        return mask.reshape(h, w)

    if mode == "percentile":
        thr = np.percentile(flat, float(FIX_PERCENTILE))
        return (gaze_map_01 >= thr).astype(np.uint8)

    raise ValueError(f"Unknown FIXATION_MODE='{FIXATION_MODE}'. Expected nonzero/topk/percentile.")


def metric_auc(model_map: np.ndarray, fix_mask: np.ndarray, rng: Optional[np.random.RandomState] = None) -> float:
    y_true = fix_mask.reshape(-1).astype(np.uint8)
    if int(y_true.sum()) == 0 or int(y_true.sum()) == int(y_true.size):
        return np.nan
    y_score = model_map.reshape(-1).astype(np.float32)
    neg_idx = np.where(y_true == 0)[0]
    pos_idx = np.where(y_true == 1)[0]
    n_neg = min(int(AUC_NEG_SAMPLES), int(neg_idx.size))
    if rng is not None and n_neg < int(neg_idx.size):
        neg_idx = rng.choice(neg_idx, size=n_neg, replace=False)
    idx = np.concatenate([pos_idx, neg_idx])
    y_true = y_true[idx]
    y_score = y_score[idx]
    return float(roc_auc_score(y_true, y_score))


def metric_sauc(model_map: np.ndarray, fix_mask: np.ndarray, neg_pool_xy: np.ndarray, rng: np.random.RandomState) -> float:
    pos_xy = np.argwhere(fix_mask > 0)
    if pos_xy.shape[0] == 0:
        return np.nan
    if neg_pool_xy.shape[0] == 0:
        return np.nan

    n_neg = int(min(int(sAUC_NEG_SAMPLES), int(neg_pool_xy.shape[0])))
    neg_idx = rng.choice(neg_pool_xy.shape[0], size=n_neg, replace=False)
    neg_xy = neg_pool_xy[neg_idx]

    pos_scores = model_map[pos_xy[:, 0], pos_xy[:, 1]].astype(np.float32)
    neg_scores = model_map[neg_xy[:, 0], neg_xy[:, 1]].astype(np.float32)

    y_true = np.concatenate([np.ones_like(pos_scores), np.zeros_like(neg_scores)]).astype(np.uint8)
    y_score = np.concatenate([pos_scores, neg_scores]).astype(np.float32)

    if y_true.sum() == 0 or y_true.sum() == y_true.size:
        return np.nan
    return float(roc_auc_score(y_true, y_score))


def metric_nss(model_map: np.ndarray, fix_mask: np.ndarray) -> float:
    z = _zscore(model_map)
    pts = z[fix_mask > 0]
    if pts.size == 0:
        return np.nan
    return float(np.mean(pts))


def metric_cc(model_map: np.ndarray, gaze_map: np.ndarray) -> float:
    a = _zscore(model_map).reshape(-1).astype(np.float64)
    b = _zscore(gaze_map).reshape(-1).astype(np.float64)
    sa = float(np.std(a))
    sb = float(np.std(b))
    if sa < 1e-12 or sb < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def metric_emd(model_map: np.ndarray, gaze_map: np.ndarray) -> float:
    p = _prob_norm(model_map)
    q = _prob_norm(gaze_map)
    h, w = p.shape
    xs = np.arange(w, dtype=np.float64)
    ys = np.arange(h, dtype=np.float64)
    emd_x = wasserstein_distance(xs, xs, u_weights=p.sum(axis=0), v_weights=q.sum(axis=0))
    emd_y = wasserstein_distance(ys, ys, u_weights=p.sum(axis=1), v_weights=q.sum(axis=1))
    return float(emd_x + emd_y)


def metric_sim(model_map: np.ndarray, gaze_map: np.ndarray) -> float:
    p = _prob_norm(model_map).reshape(-1)
    q = _prob_norm(gaze_map).reshape(-1)
    return float(np.sum(np.minimum(p, q)))


def metric_kl(model_map: np.ndarray, gaze_map: np.ndarray, eps: float = 1e-12) -> float:
    p = _prob_norm(gaze_map).reshape(-1)  # gaze distribution
    q = _prob_norm(model_map).reshape(-1) # model distribution
    q = np.clip(q, eps, None)
    p = np.clip(p, eps, None)
    return float(np.sum(p * (np.log(p) - np.log(q))))


def metric_ig(model_map: np.ndarray, fix_mask: np.ndarray, eps: float = 1e-12) -> float:
    p_model = _prob_norm(model_map).astype(np.float64)
    h, w = p_model.shape
    p_base = np.ones((h, w), dtype=np.float64) / float(h * w)

    pos = np.argwhere(fix_mask > 0)
    if pos.shape[0] == 0:
        return np.nan

    lm = np.log2(np.clip(p_model[pos[:, 0], pos[:, 1]], eps, None))
    lb = np.log2(np.clip(p_base[pos[:, 0], pos[:, 1]], eps, None))
    return float(np.mean(lm - lb))


@torch.no_grad()
def evaluate_saliency_metrics(net: torch.nn.Module, loader: DataLoader, seed: int = 0) -> Dict[str, float]:
    rng = np.random.RandomState(int(seed) + 999)

    # Build negative fixation pool for sAUC from gaze maps in the loader
    neg_pool = []
    if bool(COMPUTE_sAUC):
        for batch in loader:
            gL = _to_2d(batch.get("gaze_l"))
            gR = _to_2d(batch.get("gaze_r"))
            okL = batch.get("gaze_ok_l", None)
            okR = batch.get("gaze_ok_r", None)

            if gL is not None and okL is not None:
                for i in range(gL.shape[0]):
                    if bool(okL[i].item()):
                        gm = gL[i].cpu().numpy()
                        gm01 = _minmax01(gm)
                        fm = _fixation_mask_from_gaze(gm01)
                        xy = np.argwhere(fm > 0)
                        if xy.shape[0] > 0:
                            neg_pool.append(xy)

            if gR is not None and okR is not None:
                for i in range(gR.shape[0]):
                    if bool(okR[i].item()):
                        gm = gR[i].cpu().numpy()
                        gm01 = _minmax01(gm)
                        fm = _fixation_mask_from_gaze(gm01)
                        xy = np.argwhere(fm > 0)
                        if xy.shape[0] > 0:
                            neg_pool.append(xy)

        neg_pool_xy = np.concatenate(neg_pool, axis=0) if len(neg_pool) else np.zeros((0, 2), dtype=np.int64)
    else:
        neg_pool_xy = np.zeros((0, 2), dtype=np.int64)

    sums = {k: 0.0 for k in ["AUC", "sAUC", "NSS", "CC", "EMD", "SIM", "KL", "IG"]}
    counts = {k: 0 for k in sums.keys()}

    for batch in loader:
        out = forward_model_matching_train(net, batch)
        aL = out["left"].get("attn_map", None)
        aR = out["right"].get("attn_map", None)

        gL = _to_2d(batch.get("gaze_l", None))
        gR = _to_2d(batch.get("gaze_r", None))
        okL = batch.get("gaze_ok_l", None)
        okR = batch.get("gaze_ok_r", None)

        # Left branch
        if aL is not None and gL is not None and okL is not None:
            aL2 = _to_2d(aL).detach().cpu().numpy()
            gL2 = gL.detach().cpu().numpy()

            for i in range(aL2.shape[0]):
                if not bool(okL[i].item()):
                    continue

                mm = _minmax01(aL2[i])
                gg = gL2[i].astype(np.float32)
                gg01 = _minmax01(gg)

                fm = _fixation_mask_from_gaze(gg01)

                v_auc = metric_auc(mm, fm, rng)
                if not np.isnan(v_auc):
                    sums["AUC"] += float(v_auc); counts["AUC"] += 1

                if bool(COMPUTE_sAUC):
                    v_sauc = metric_sauc(mm, fm, neg_pool_xy, rng)
                    if not np.isnan(v_sauc):
                        sums["sAUC"] += float(v_sauc); counts["sAUC"] += 1

                v_nss = metric_nss(mm, fm)
                if not np.isnan(v_nss):
                    sums["NSS"] += float(v_nss); counts["NSS"] += 1

                v_cc = metric_cc(mm, gg01)
                if not np.isnan(v_cc):
                    sums["CC"] += float(v_cc); counts["CC"] += 1

                v_emd = metric_emd(mm, gg01)
                if not np.isnan(v_emd):
                    sums["EMD"] += float(v_emd); counts["EMD"] += 1

                v_sim = metric_sim(mm, gg01)
                if not np.isnan(v_sim):
                    sums["SIM"] += float(v_sim); counts["SIM"] += 1

                v_kl = metric_kl(mm, gg01)
                if not np.isnan(v_kl):
                    sums["KL"] += float(v_kl); counts["KL"] += 1

                v_ig = metric_ig(mm, fm)
                if not np.isnan(v_ig):
                    sums["IG"] += float(v_ig); counts["IG"] += 1

        # Right branch
        if aR is not None and gR is not None and okR is not None:
            aR2 = _to_2d(aR).detach().cpu().numpy()
            gR2 = gR.detach().cpu().numpy()

            for i in range(aR2.shape[0]):
                if not bool(okR[i].item()):
                    continue

                mm = _minmax01(aR2[i])
                gg = gR2[i].astype(np.float32)
                gg01 = _minmax01(gg)

                fm = _fixation_mask_from_gaze(gg01)

                v_auc = metric_auc(mm, fm, rng)
                if not np.isnan(v_auc):
                    sums["AUC"] += float(v_auc); counts["AUC"] += 1

                if bool(COMPUTE_sAUC):
                    v_sauc = metric_sauc(mm, fm, neg_pool_xy, rng)
                    if not np.isnan(v_sauc):
                        sums["sAUC"] += float(v_sauc); counts["sAUC"] += 1

                v_nss = metric_nss(mm, fm)
                if not np.isnan(v_nss):
                    sums["NSS"] += float(v_nss); counts["NSS"] += 1

                v_cc = metric_cc(mm, gg01)
                if not np.isnan(v_cc):
                    sums["CC"] += float(v_cc); counts["CC"] += 1

                v_emd = metric_emd(mm, gg01)
                if not np.isnan(v_emd):
                    sums["EMD"] += float(v_emd); counts["EMD"] += 1

                v_sim = metric_sim(mm, gg01)
                if not np.isnan(v_sim):
                    sums["SIM"] += float(v_sim); counts["SIM"] += 1

                v_kl = metric_kl(mm, gg01)
                if not np.isnan(v_kl):
                    sums["KL"] += float(v_kl); counts["KL"] += 1

                v_ig = metric_ig(mm, fm)
                if not np.isnan(v_ig):
                    sums["IG"] += float(v_ig); counts["IG"] += 1

    out = {}
    for k in sums.keys():
        out[k] = float(sums[k] / float(max(1, counts[k]))) if counts[k] > 0 else np.nan
    out["n_samples_used"] = int(max(counts.values()) if counts else 0)
    return out


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
import numpy as np
import pandas as pd

# Uses COMPUTE_sAUC and sAUC_NEG_SAMPLES from the configuration cell above.

def eval_saliency_table_chunked() -> pd.DataFrame:
    """
    Evaluates saliency metrics iteratively to prevent VRAM OOM.
    Moves all models to CPU first, then loads one to GPU -> Evaluates -> Deletes.
    """
    # ---------------------------------------------------------
    # STEP 1: EVICT ALL MODELS TO CPU TO FREE VRAM IMMEDIATELY
    # ---------------------------------------------------------
    print("Moving all models to CPU to clear VRAM...")
    for tag, pack in MODELS.items():
        if "net" in pack and pack["net"] is not None:
            pack["net"].cpu()
            
    gc.collect()
    torch.cuda.empty_cache()
    print("VRAM cleared. Starting evaluation loop...\n")

    rows = []
    # Convert keys to a list so we can mutate the dictionary during iteration
    tags = list(MODELS.keys())

    for tag in tags:
        print(f"Evaluating {tag}...")
        pack = MODELS[tag]
        rr = pack["run"]
        net = pack["net"]
        test_loader = LOADERS[tag]["test"]
        seed = int(getattr(rr.args, "seed", 0))

        # ---------------------------------------------------------
        # STEP 2: LOAD ONLY THE CURRENT MODEL TO GPU
        # ---------------------------------------------------------
        net.to(DEVICE)

        has_cfg = hasattr(net, "cfg") and hasattr(net.cfg, "attention")
        orig_mode = net.cfg.attention.mode if has_cfg else "raw"

        # --- Evaluate RAW ---
        if hasattr(net, "attn_recorder") and hasattr(net.attn_recorder, "reset"):
            net.attn_recorder.reset()
        if has_cfg:
            object.__setattr__(net.cfg.attention, "mode", "raw")
            
        raw_metrics = evaluate_saliency_metrics(net, test_loader, seed=seed)

        # Mid-cleanup
        if hasattr(net, "attn_recorder") and hasattr(net.attn_recorder, "reset"):
            net.attn_recorder.reset()
        gc.collect()
        torch.cuda.empty_cache()

        # --- Evaluate ROLLOUT ---
        if has_cfg:
            object.__setattr__(net.cfg.attention, "mode", "rollout")
            
        rollout_metrics = evaluate_saliency_metrics(net, test_loader, seed=seed)

        # Build the row dictionary
        row = {
            "tag": tag,
            "wandb_run_id": rr.run_id or "",
            "attention_mode(eval)": "raw / rollout",
            "attn_layer(eval)": int(getattr(net.cfg.attention, "layer", -999)) if has_cfg else -999,
        }

        for k in ["AUC", "sAUC", "NSS", "CC", "EMD", "SIM", "KL", "IG"]:
            val_raw = raw_metrics.get(k, np.nan)
            val_roll = rollout_metrics.get(k, np.nan)
            row[f"test_{k}"] = f"{val_raw:.3f} / {val_roll:.3f}"

        row["test_n_samples_used"] = raw_metrics.get("n_samples_used", 0)
        rows.append(row)

        # ---------------------------------------------------------
        # STEP 3: CLEANUP FOR THE NEXT ITERATION
        # ---------------------------------------------------------
        if has_cfg:
            object.__setattr__(net.cfg.attention, "mode", orig_mode)
        print(f"Finished {tag}. Moving model back to CPU...")
        net.cpu()          # Keep the model object available for later optional cells
        
        gc.collect()
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)

sal_df = eval_saliency_table_chunked()
display(sal_df)

## Optional: save tables to CSV

In [ ]:
OUT_DIR = Path("./notebook_outputs").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

acc_path = OUT_DIR / "checkpoint_accuracy_table.csv"
sal_path = OUT_DIR / "checkpoint_saliency_table.csv"

if "acc_df" in globals():
    acc_df.to_csv(acc_path, index=False)
else:
    print("Skipping accuracy CSV because acc_df has not been computed. Run eval_accuracy_table_full() first.")

if "sal_df" in globals():
    sal_df.to_csv(sal_path, index=False)
else:
    print("Skipping saliency CSV because sal_df has not been computed.")

print("Wrote:")
print(" -", acc_path)
print(" -", sal_path)


In [ ]:
import numpy as np
import pandas as pd

def generate_latex_accuracy(df_num):
    """
    Takes the numeric DataFrame returned by eval_accuracy_table_full()
    and prints the LaTeX table for the paper (4-City Average).
    """
    def parse_tag(t):
        t = str(t).lower()
        if 'dinov3' in t: bb = 'DINOv3'
        elif 'deit' in t: bb = 'DeiT III'
        elif 'clip' in t: bb = 'CLIP'
        else: bb = 'Unknown'
        
        if 'baseline' in t: met = 'Baseline'
        elif 'guide' in t: met = 'GII injection'
        elif 'egvit' in t: met = 'EGViT'
        elif 'align' in t: met = 'EG-PCS-Net'
        else: met = 'Unknown'
        return bb, met

    records = []
    for i in range(len(df_num)):
        row = df_num.iloc[i]
        tag = row[('Model', '')]
        bb, met = parse_tag(tag)
        
        # Extract Test and City specific data
        r_test = row.get(('Test', 'Rank'), np.nan)
        c_test = row.get(('Test', 'Class'), np.nan)
        
        # Handle potential variations in casing for the city columns
        r_paris = row.get(('City-wise', 'paris Rank'), row.get(('City-wise', 'Paris Rank'), np.nan))
        c_paris = row.get(('City-wise', 'paris Class'), row.get(('City-wise', 'Paris Class'), np.nan))
        
        r_munich = row.get(('City-wise', 'munich Rank'), row.get(('City-wise', 'Munich Rank'), np.nan))
        c_munich = row.get(('City-wise', 'munich Class'), row.get(('City-wise', 'Munich Class'), np.nan))
        
        r_barca = row.get(('City-wise', 'barcelona Rank'), row.get(('City-wise', 'Barcelona Rank'), np.nan))
        c_barca = row.get(('City-wise', 'barcelona Class'), row.get(('City-wise', 'Barcelona Class'), np.nan))
        
        # Calculate 4-City Average dynamically (including Test/Berlin)
        r_avg = np.nanmean([r_test, r_paris, r_munich, r_barca])
        c_avg = np.nanmean([c_test, c_paris, c_munich, c_barca])
        
        records.append({
            'Backbone': bb, 'Method': met,
            'Test_R': r_test * 100, 'Test_C': c_test * 100,
            'Paris_R': r_paris * 100, 'Paris_C': c_paris * 100,
            'Munich_R': r_munich * 100, 'Munich_C': c_munich * 100,
            'Barca_R': r_barca * 100, 'Barca_C': c_barca * 100,
            'Avg_R': r_avg * 100, 'Avg_C': c_avg * 100
        })
        
    df_p = pd.DataFrame(records)
    
    # Calculate best metrics per backbone for bolding
    best = {}
    for bb in ['DINOv3', 'DeiT III', 'CLIP']:
        sub = df_p[df_p['Backbone'] == bb]
        if len(sub) > 0:
            best[bb] = {
                'Test_R': sub['Test_R'].max(), 'Test_C': sub['Test_C'].max(),
                'Paris_R': sub['Paris_R'].max(), 'Paris_C': sub['Paris_C'].max(),
                'Munich_R': sub['Munich_R'].max(), 'Munich_C': sub['Munich_C'].max(),
                'Barca_R': sub['Barca_R'].max(), 'Barca_C': sub['Barca_C'].max(),
                'Avg_R': sub['Avg_R'].max(), 'Avg_C': sub['Avg_C'].max(),
            }
        
    # Build LaTeX String
    lines = [
        "\\begin{table*}[t]",
        "\\centering",
        "\\caption{Best-run predictive performance (Acc.\\ Rank/Class). ``Test'' refers to the Berlin test split. The last column averages the performance across all 4 cities.}",
        "\\label{tab:cross_city_best}",
        "\\setlength{\\tabcolsep}{6pt}",
        "\\begin{tabular}{llccccc}",
        "\\toprule",
        "Backbone & Method & Test (Berlin) & Paris & Munich & Barcelona & 4-City Avg \\\\",
        "\\midrule"
    ]
    
    for bb in ['DINOv3', 'DeiT III', 'CLIP']:
        lines.append(f"\\multirow{{4}}{{*}}{{{bb}}} ")
        for met in ['Baseline', 'GII injection', 'EGViT', 'EG-PCS-Net']:
            sub = df_p[(df_p['Backbone'] == bb) & (df_p['Method'] == met)]
            if len(sub) == 0: 
                continue
            row = sub.iloc[0]
            
            def fmt(val, best_val):
                if pd.isna(val): return "-"
                s = f"{val:.2f}"
                if abs(val - best_val) < 1e-5: return f"\\textbf{{{s}}}"
                return s
            
            test_str = f"{fmt(row['Test_R'], best[bb]['Test_R'])}/{fmt(row['Test_C'], best[bb]['Test_C'])}"
            paris_str = f"{fmt(row['Paris_R'], best[bb]['Paris_R'])}/{fmt(row['Paris_C'], best[bb]['Paris_C'])}"
            munich_str = f"{fmt(row['Munich_R'], best[bb]['Munich_R'])}/{fmt(row['Munich_C'], best[bb]['Munich_C'])}"
            barca_str = f"{fmt(row['Barca_R'], best[bb]['Barca_R'])}/{fmt(row['Barca_C'], best[bb]['Barca_C'])}"
            avg_str = f"{fmt(row['Avg_R'], best[bb]['Avg_R'])}/{fmt(row['Avg_C'], best[bb]['Avg_C'])}"
            
            lines.append(f"& {met} & {test_str} & {paris_str} & {munich_str} & {barca_str} & {avg_str} \\\\")
        
        if bb != 'CLIP':
            lines.append("\\midrule")
            
    lines.extend([
        "\\bottomrule",
        "\\end{tabular}",
        "\\end{table*}"
    ])
    
    print("\n".join(lines))

# Generate the accuracy table LaTeX
generate_latex_accuracy(acc_df)

In [ ]:
def generate_latex_saliency(sal_df):
    """
    Takes the DataFrame returned by eval_saliency_table()
    and prints the corresponding LaTeX table with raw / rollout format.
    """
    def parse_tag(t):
        t = str(t).lower()
        if 'dinov3' in t: bb = 'DINOv3'
        elif 'deit' in t: bb = 'DeiT III'
        elif 'clip' in t: bb = 'CLIP'
        else: bb = 'Unknown'
        
        # Distinguish between the two trained alignment models
        if 'baseline' in t: met = 'Baseline'
        elif 'guide' in t: met = 'GII injection'
        elif 'egvit' in t: met = 'EGViT'
        elif 'align_raw' in t: met = 'EG-PCS-Net (Raw)'
        elif 'align_rollout' in t: met = 'EG-PCS-Net (Rollout)'
        elif 'align' in t: met = 'EG-PCS-Net' # Fallback
        else: met = 'Unknown'
        return bb, met
        
    metrics = ['test_AUC', 'test_NSS', 'test_CC', 'test_EMD', 'test_SIM', 'test_KL', 'test_IG']
    directions = {'test_AUC': 1, 'test_NSS': 1, 'test_CC': 1, 'test_EMD': -1, 'test_SIM': 1, 'test_KL': -1, 'test_IG': 1}
    
    # 1. Parse the strings "raw / rollout" into separate floats
    records = []
    for i, row in sal_df.iterrows():
        bb, met = parse_tag(row['tag'])
        rec = {'Backbone': bb, 'Method': met}
        for m in metrics:
            val_str = row.get(m, "nan / nan")
            if isinstance(val_str, str) and " / " in val_str:
                parts = val_str.split(" / ")
                try:
                    rec[m + '_raw'] = float(parts[0])
                    rec[m + '_roll'] = float(parts[1])
                except ValueError:
                    rec[m + '_raw'], rec[m + '_roll'] = np.nan, np.nan
            else:
                rec[m + '_raw'], rec[m + '_roll'] = np.nan, np.nan
        records.append(rec)
        
    df_p = pd.DataFrame(records)
    
    # 2. Calculate best metrics per backbone for raw and rollout independently
    best = {}
    for bb in ['DINOv3', 'DeiT III', 'CLIP']:
        sub = df_p[df_p['Backbone'] == bb]
        if len(sub) > 0:
            best[bb] = {}
            for m in metrics:
                m_raw = m + '_raw'
                m_roll = m + '_roll'
                if directions[m] == 1:
                    best[bb][m_raw] = sub[m_raw].max()
                    best[bb][m_roll] = sub[m_roll].max()
                else:
                    best[bb][m_raw] = sub[m_raw].min()
                    best[bb][m_roll] = sub[m_roll].min()
                
    # 3. Build LaTeX String
    lines = [
        "\\begin{table*}[t]",
        "\\centering",
        "\\caption{Saliency evaluation on the test set (Raw / Rollout). $\\uparrow$ indicates higher is better, $\\downarrow$ indicates lower is better.}",
        "\\label{tab:saliency_results}",
        "\\setlength{\\tabcolsep}{4pt}",
        "\\begin{tabular}{llccccccc}",
        "\\toprule",
        "Backbone & Method & AUC$\\uparrow$ & NSS$\\uparrow$ & CC$\\uparrow$ & EMD$\\downarrow$ & SIM$\\uparrow$ & KL$\\downarrow$ & IG$\\uparrow$ \\\\",
        "\\midrule"
    ]
    
    for bb in ['DINOv3', 'DeiT III', 'CLIP']:
        # Changed to 5 rows to account for both Align model variants
        lines.append(f"\\multirow{{5}}{{*}}{{{bb}}} ") 
        
        # Added both EG-PCS-Net variants to the layout loop
        methods = ['Baseline', 'GII injection', 'EGViT', 'EG-PCS-Net (Raw)', 'EG-PCS-Net (Rollout)']
        
        for met in methods:
            sub = df_p[(df_p['Backbone'] == bb) & (df_p['Method'] == met)]
            if len(sub) == 0: 
                continue
            row = sub.iloc[0]
            
            vals = []
            for m in metrics:
                v_raw = row[m + '_raw']
                v_roll = row[m + '_roll']
                
                if pd.isna(v_raw) or pd.isna(v_roll):
                    vals.append("- / -")
                    continue
                
                # Format to 3 decimal places
                s_raw = f"{v_raw:.3f}"
                s_roll = f"{v_roll:.3f}"
                
                # Apply bolding independently
                if abs(v_raw - best[bb][m + '_raw']) < 1e-6:
                    s_raw = f"\\textbf{{{s_raw}}}"
                if abs(v_roll - best[bb][m + '_roll']) < 1e-6:
                    s_roll = f"\\textbf{{{s_roll}}}"
                    
                # Reconstruct the string
                vals.append(f"{s_raw} / {s_roll}")
                
            lines.append(f"& {met} & {' & '.join(vals)} \\\\")
            
        if bb != 'CLIP':
            lines.append("\\midrule")
            
    lines.extend([
        "\\bottomrule",
        "\\end{tabular}",
        "\\end{table*}"
    ])
    
    print("\n".join(lines))

# Generate the saliency table LaTeX
generate_latex_saliency(sal_df)

## Optional: save tables to CSV

In [ ]:
# ------------------------------------------------------------
# Histogram: rank margin frequency (test split) for a chosen checkpoint tag
# - Rank margin = sL - sR (continuous)
# - Shows min/max of margins
# - Histogram bins have width 0.1
# ------------------------------------------------------------

CHECKPOINT_TAG_FOR_RANK_HIST = "dinov3_align_raw"  # Must exist in MODELS and LOADERS; use sorted(MODELS) to inspect valid tags

if CHECKPOINT_TAG_FOR_RANK_HIST not in LOADERS or CHECKPOINT_TAG_FOR_RANK_HIST not in MODELS:
    raise KeyError(
        f"Unknown tag '{CHECKPOINT_TAG_FOR_RANK_HIST}'. Available: {sorted(list(LOADERS.keys()))}"
    )

net = MODELS[CHECKPOINT_TAG_FOR_RANK_HIST]["net"]
test_loader = LOADERS[CHECKPOINT_TAG_FOR_RANK_HIST]["test"]

margins = []

with torch.no_grad():
    for batch in test_loader:
        out = forward_model_matching_train(net, batch)
        sL = out["left"]["output"].view(-1)
        sR = out["right"]["output"].view(-1)

        m = (sL - sR).detach().cpu().numpy().astype(np.float32)
        margins.append(m)

margins = np.concatenate(margins, axis=0) if len(margins) else np.zeros((0,), dtype=np.float32)

if margins.size == 0:
    raise RuntimeError("No margins collected from test_loader.")

mn = float(np.min(margins))
mx = float(np.max(margins))

bin_w = 0.1
left = math.floor(mn / bin_w) * bin_w
right = math.ceil(mx / bin_w) * bin_w
bins = np.arange(left, right + bin_w, bin_w)

plt.figure(figsize=(7, 4))
plt.hist(margins, bins=bins)
plt.title(f"Rank margin histogram (sL - sR), bin=0.1 — {CHECKPOINT_TAG_FOR_RANK_HIST}\nmin={mn:.4f}, max={mx:.4f}, N={margins.size}")
plt.xlabel("rank margin (sL - sR)")
plt.ylabel("count")
plt.show()

print(f"min margin: {mn:.6f}")
print(f"max margin: {mx:.6f}")
print(f"N margins:  {margins.size}")
print(f"bins: {bins[0]:.2f} .. {bins[-1]:.2f} (step {bin_w})")
